# 02 — Retornos y distribución empírica: preparación de entradas

Esta notebook prepara y valida las entradas locales del corte final antes de cualquier transformación o análisis estadístico. Carga la vista elegible de precios ajustados Gold y la matriz de retornos simples correspondiente. No descarga datos, no usa credenciales y no implementa todavía distribución empírica, diagnósticos, escenarios ni carteras.

## Alcance actual

El corte aprobado cubre 2018-01-02 a 2026-10-09: 2,205 fechas de precios y 2,204 retornos diarios utilizables por instrumento. La matriz Gold de precios conserva 138 series para auditoría; aquí se selecciona explícitamente, y sin persistirla, la vista de las 125 columnas que pertenecen al universo elegible.

La primera fila de retornos se conserva durante la validación porque representa la ausencia prevista de un precio anterior aprobado. Sólo después se crea una vista de retornos utilizables retirando esa fila conocida; no se usa `drop_nulls` ni se imputan datos.

In [1]:
from __future__ import annotations

import hashlib
import json
import site
import sys
from datetime import date
from pathlib import Path

def find_project_root() -> Path:
    def is_project_root(candidate: Path) -> bool:
        return ((candidate / 'config' / 'universe.csv').is_file()
                and (candidate / 'src' / 'scenario_portfolio_lab').is_dir())

    for origin in (Path.cwd(), *Path.cwd().parents):
        for candidate in (origin, origin / 'scenario-portfolio-lab', origin / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if is_project_root(candidate):
                return candidate
    raise RuntimeError(
        'Repository files config/universe.csv and src/scenario_portfolio_lab were not found. '
        'Open the notebook from the complete scenario-portfolio-lab repository (or its data-science-portfolio parent). '
        'Do not run it from an exported standalone .ipynb file.'
    )


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# MyST/Jupyter kernels may differ from the project virtual environment.
# Reuse already installed local dependencies when this Windows project .venv exists; never install packages here.
LOCAL_SITE_PACKAGES = ROOT / '.venv' / 'Lib' / 'site-packages'
if LOCAL_SITE_PACKAGES.is_dir():
    site.addsitedir(str(LOCAL_SITE_PACKAGES))

try:
    import exchange_calendars as xcals
    import polars as pl
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'Missing project dependencies. Select the project .venv Python 3.13.2 kernel or install requirements.txt '
        'into the active kernel environment before running this notebook.'
    ) from exc

RUN_ROOT = ROOT / 'data' / 'published' / 'cut_2026-10-09_run_2026-10-11_full_window_final'
GOLD_ROOT = RUN_ROOT / 'gold'
PRICE_MATRIX_PATH = GOLD_ROOT / 'adjusted_close_matrix.parquet'
RETURN_MATRIX_PATH = GOLD_ROOT / 'eligible_simple_total_return_matrix.parquet'
RUN_MANIFEST_PATH = RUN_ROOT / 'run_manifest.json'
ELIGIBLE_CATALOG_PATH = RUN_ROOT / 'reports' / 'eligible_universe.csv'

for required in (PRICE_MATRIX_PATH, RETURN_MATRIX_PATH, RUN_MANIFEST_PATH, ELIGIBLE_CATALOG_PATH):
    if not required.is_file():
        raise FileNotFoundError(f'Required final-cut artifact is missing: {required.relative_to(ROOT)}')


## Carga del manifiesto y catálogo elegible

El manifiesto final contiene hashes de los 138 raw que alimentaron el corte. Esta notebook los comprueba contra sus referencias previas. El manifiesto no contiene hashes de referencia para los dos Parquet matriciales; por ello valida su estructura y contenido, pero no los presenta como verificados criptográficamente contra una referencia previa.

In [2]:
run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding='utf-8'))
approved_period = run_manifest['approved_period']
assert approved_period == {'start': '2018-01-02', 'end': '2026-10-09'}

eligible_catalog = pl.read_csv(ELIGIBLE_CATALOG_PATH).filter(pl.col('universe_status') == 'ELIGIBLE')
eligible_symbols = eligible_catalog['symbol'].to_list()
assert len(eligible_symbols) == 125 and len(set(eligible_symbols)) == 125

raw_hash_checks = []
for symbol, reference in run_manifest['raw'].items():
    raw_path = ROOT / Path(reference['path'].replace('\\', '/'))
    if not raw_path.is_file():
        raise FileNotFoundError(f'Manifest-referenced raw is missing for {symbol}.')
    raw_hash_checks.append(hashlib.sha256(raw_path.read_bytes()).hexdigest() == reference['sha256'])
assert len(raw_hash_checks) == 138 and all(raw_hash_checks)


## Carga y validación de matrices

Las sesiones esperadas se reconstruyen desde el calendario común documentado en el manifiesto (`exchange_calendars:US_EQUITY_COMMON(XNYS=XNAS=ARCX)`). Las validaciones son explícitas: fechas, orden de símbolos, precios, nulos previstos en retornos y finitud.

In [3]:
all_gold_prices = pl.read_parquet(PRICE_MATRIX_PATH)
return_matrix = pl.read_parquet(RETURN_MATRIX_PATH)

assert all_gold_prices.columns[0] == return_matrix.columns[0] == 'session_date'
return_symbols = return_matrix.columns[1:]
assert return_symbols == eligible_symbols
assert set(return_symbols).issubset(set(all_gold_prices.columns[1:]))

# Explicit ordered view: not a new persisted matrix and not a silent filter.
eligible_prices = all_gold_prices.select(['session_date', *return_symbols])
assert eligible_prices.width == return_matrix.width == 126
assert eligible_prices.height == return_matrix.height == 2205

start, end = date.fromisoformat(approved_period['start']), date.fromisoformat(approved_period['end'])
expected_dates = [stamp.date() for stamp in xcals.get_calendar('XNYS').sessions if start <= stamp.date() <= end]
for matrix in (eligible_prices, return_matrix):
    dates = matrix['session_date'].to_list()
    assert dates == expected_dates
    assert dates == sorted(dates) and len(set(dates)) == len(dates) == 2205

price_values = eligible_prices.select(pl.exclude('session_date'))
assert all(value == 0 for value in price_values.null_count().row(0))
assert not any(price_values.select(pl.all().is_nan().any()).row(0))
assert not any(price_values.select(pl.all().is_infinite().any()).row(0))
assert not any(price_values.select((pl.all() <= 0).any()).row(0))

first_return_row = return_matrix.slice(0, 1).select(pl.exclude('session_date'))
subsequent_return_rows = return_matrix.slice(1).select(pl.exclude('session_date'))
assert all(value == 1 for value in return_matrix.select(pl.exclude('session_date').null_count()).row(0))
assert all(value == 1 for value in first_return_row.null_count().row(0))
assert all(value == 0 for value in subsequent_return_rows.null_count().row(0))
assert not any(subsequent_return_rows.select(pl.all().is_nan().any()).row(0))
assert not any(subsequent_return_rows.select(pl.all().is_infinite().any()).row(0))


## Vista explícita de retornos utilizables

La siguiente vista retira solamente la primera fila prevista, ya comprobada como nula para los 125 instrumentos. Conserva las fechas y el orden explícito de símbolos de la matriz elegible. No se calculan transformaciones ni estadísticas.

In [4]:
usable_returns = return_matrix.slice(1).select(['session_date', *return_symbols])
assert usable_returns.shape == (2204, 126)
assert usable_returns['session_date'].to_list() == expected_dates[1:]
assert usable_returns.columns[1:] == return_symbols
assert all(value == 0 for value in usable_returns.select(pl.exclude('session_date').null_count()).row(0))

validation_summary = pl.DataFrame({
    'check': [
        'Approved period',
        'Expected and observed price dates',
        'Eligible symbols',
        'Eligible adjusted-price view',
        'Eligible simple-return matrix',
        'Usable-return view',
        'Manifest-referenced raw hashes',
        'Matrix reference hashes',
    ],
    'result': [
        '2018-01-02 to 2026-10-09',
        '2,205 ordered, unique XNYS common sessions',
        '125 exact symbols, explicit order',
        '2,205 rows × (date + 125); positive, finite, no nulls',
        '2,205 rows × (date + 125); first-row null only, finite thereafter',
        '2,204 rows × (date + 125); dates retained, no nulls',
        '138/138 verified against manifest references',
        'Not available in run_manifest; no new hash is treated as prior evidence',
    ],
})
validation_summary


check,result
str,str
"""Approved period""","""2018-01-02 to 2026-10-09"""
"""Expected and observed price da…","""2,205 ordered, unique XNYS com…"
"""Eligible symbols""","""125 exact symbols, explicit or…"
"""Eligible adjusted-price view""","""2,205 rows × (date + 125); pos…"
"""Eligible simple-return matrix""","""2,205 rows × (date + 125); fir…"
"""Usable-return view""","""2,204 rows × (date + 125); dat…"
"""Manifest-referenced raw hashes""","""138/138 verified against manif…"
"""Matrix reference hashes""","""Not available in run_manifest;…"


## Siguiente paso: transformaciones de retornos

Pendiente de desarrollar con el usuario. Esta notebook se detiene después de la carga y validación de entradas.